# Dynamic case generation (offshore-waves forcing)

Workflow:

1. Load **goals** from the case JSON → `simulations.goals_file`.
2. Open a **processed offshore-waves** zarr (`offshore_waves` in the case JSON).
3. Build SnapWave boundary forcing per timestep (`hs`, `tp`, `wd`, `ds`).
4. Run cases (optional) and postprocess to a single time-series NetCDF.

Processed offshore data must use dims `(time, goal)` with bulk variables
`hs`, `tp`, `dir`, `spr` (see `07_Download_Process_WHACS.ipynb`).

In [ ]:
from bluemath_tk.wrappers.snapwave import (
    SnapWaveDynamicModelWrapper,
    boundary_time_series,
)
from utils.config import (
    PATHS,
    SIMULATIONS,
    get_offshore_wave,
    load_goals,
    open_offshore_waves,
    resolve,
)
from utils import simulation as sim

## Configuration

In [ ]:
offshore_name = "WHACS"  # must match an entry in the case JSON → offshore_waves
period = "2020"  # time slice passed to xarray .sel(time=...)

offshore_entry = get_offshore_wave(offshore_name)
templates_dir = sim.templates_dir("dynamic")
cases_dir = resolve(SIMULATIONS["dynamic_cases"])
aoi_file = resolve(PATHS["aoi_file"])
mesh_file = resolve(PATHS["mesh_file"])
goals_file = resolve(SIMULATIONS["goals_file"])
offshore_path = resolve(offshore_entry["data_path"])
raw_goals = load_goals()

print(f"offshore  : {offshore_name} → {offshore_path}")
print(f"goals     : {goals_file}")
print(f"templates : {templates_dir}")
print(f"cases     : {cases_dir}")
print(f"mesh      : {mesh_file}")
print(f"period    : {period}")

## Offshore waves → boundary forcing

In [ ]:
offshore_ds = open_offshore_waves(entry=offshore_entry).sel(time=period).load()
offshore_ds

In [ ]:
# One case per time step: per-goal (= boundary node) hs/tp/dir/spr lists + tref
metamodel_parameters = boundary_time_series(offshore_ds, node_dim="goal")
list(metamodel_parameters.keys())

In [ ]:
import numpy as np

n_cases = len(metamodel_parameters["tref"])
metamodel_parameters["save_map"] = np.random.choice(
    [True, False], size=n_cases, p=[0.05, 0.95]
).tolist()
sum(metamodel_parameters["save_map"]), n_cases

## SnapWave wrapper

Create the wrapper, then run `plot_configuration()` for a **pre-run diagnostic**:
left side = domain map + observed direction-range bars per goal; right side = text summary.
Each case is one timestep with **all goals forced** on the boundary.
Review this before `build_cases()`.

In [ ]:
# Case geometry: goals = boundary nodes, AoI enclosure, labelled output sites
setup = sim.SnapWaveCaseSetup.from_case(raw_goals)

wrapper = SnapWaveDynamicModelWrapper(
    templates_dir=str(templates_dir),
    metamodel_parameters=metamodel_parameters,
    fixed_parameters={"gridfile": str(mesh_file)},
    output_dir=str(cases_dir),
    **setup.wrapper_kwargs(),
)
wrapper

In [ ]:
fig = setup.plot_dynamic_configuration(
    offshore_ds,
    offshore_name=offshore_name,
    save_map=metamodel_parameters.get("save_map"),
)
# fig.savefig("dynamic_setup_check.png", dpi=150, bbox_inches="tight")

## Build, run, postprocess

In [ ]:
wrapper.load_cases(cases_name_format=lambda ctx: f"{ctx.get('tref')}")

In [ ]:
# On the GeoOcean cluster: one SLURM array task per case. Writes case_dirs.txt
# and snapwave_array.sh (module load snapwave, OMP threads, scratch cleanup)
# into the cases folder, then submits it with sbatch.
# wrapper.run_cases_slurm(max_parallel=200)
# Or locally, with snapwave on the PATH:
wrapper.run_cases_in_background(launcher="default", num_workers=1)

In [ ]:
wrapper.monitor_cases(value_counts="simple")

## Inspect finished cases

Post-run diagnostic (reads directly from the case folder):

- top row: full-width domain `tripcolor` map when `save_map=True` for that timestep
- bottom 2×2: `hs`, `tp`, `dir`, `spr` at output sites
- title/subtitle: case id + boundary forcing summary

Customize colormaps via `CasePlotStyle(cmaps={"hs": "viridis", "tp": "plasma", "dir": "twilight_shifted", "spr": "cividis"})`.

In [ ]:
from bluemath_tk.wrappers.snapwave.snapwave_plots import CasePlotStyle
from utils.viz import plot_dynamic_case

style = CasePlotStyle(
    cmaps={"hs": "viridis", "tp": "plasma", "dir": "twilight_shifted", "spr": "cividis"}
)
plot_dynamic_case(
    "SnapWave_CASES_Dynamic/20200104 040000", style=style, show_satellite=True
);

In [ ]:
result = wrapper.postprocess_cases()
out_nc = cases_dir / f"Dynamic_Cases_{offshore_name}_{period.replace('-', '')}.nc"
result.to_netcdf(out_nc)

In [ ]:
import xarray as xr

xr.open_dataset(out_nc).load()